In [1]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
URL = "https://raw.githubusercontent.com/ageron/handson-ml/refs/heads/master/datasets/housing/housing.csv"
df = pd.read_csv(URL)
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(df, test_size=0.2, random_state=35)

In [4]:
housing = train_set.drop("median_house_value", axis=1)
housing_labels = train_set["median_house_value"].copy()

In [5]:
housing_num = housing.drop("ocean_proximity", axis=1)
housing_num

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
1380,-122.09,38.00,6.0,10191.0,1882.0,4377.0,1789.0,5.2015
12294,-116.93,33.93,13.0,7804.0,1594.0,3297.0,1469.0,2.0549
7387,-118.25,33.97,37.0,794.0,210.0,814.0,213.0,2.2917
14454,-117.27,32.83,39.0,1877.0,426.0,805.0,409.0,3.8750
2927,-119.01,35.36,24.0,1941.0,484.0,1277.0,435.0,1.0560
...,...,...,...,...,...,...,...,...
19391,-120.85,37.78,25.0,421.0,NaN,303.0,106.0,2.2679
15393,-116.90,33.22,11.0,4132.0,773.0,2012.0,703.0,3.1906
9143,-117.96,34.48,32.0,1896.0,342.0,806.0,299.0,4.5769
17679,-121.84,37.32,22.0,3015.0,581.0,2491.0,530.0,4.3419


## Numbers

In [7]:
from sklearn.base import BaseEstimator, TransformerMixin

rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room = True): # no *args or **kargs
        self.add_bedrooms_per_room = add_bedrooms_per_room
    def fit(self, X, y=None):
        return self # nothing else to do
    def transform(self, X, y=None):
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
        population_per_household = X[:, population_ix] / X[:, households_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household,
                         bedrooms_per_room]
        else:
            return np.c_[X, rooms_per_household, population_per_household]

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('attribs_adder', CombinedAttributesAdder(add_bedrooms_per_room = True)),
    ('std_scaler', StandardScaler())
])

num_pipeline.fit_transform(housing_num)

array([[-1.25390838,  1.10757958, -1.79230424, ...,  0.10067299,
        -0.05831117, -0.44204612],
       [ 1.31649014, -0.79863258, -1.2367069 , ..., -0.04645309,
        -0.07680567, -0.14642693],
       [ 0.65894633, -0.77989831,  0.66819829, ..., -0.6536053 ,
         0.06743175,  0.76284898],
       ...,
       [ 0.80340671, -0.54103634,  0.27134305, ...,  0.34765451,
        -0.03553681, -0.50685707],
       [-1.12937357,  0.78909696, -0.52236745, ...,  0.09768495,
         0.14776252, -0.32081214],
       [ 0.62407658, -0.67217624,  0.58882724, ...,  0.35493053,
        -0.0368153 , -0.65791088]], shape=(16512, 11))

## Texts

In [16]:
from sklearn.compose import ColumnTransformer

num_attribs = list(housing_num)
cat_attribs = ['ocean_proximity']

In [17]:
full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_attribs),
    ('cat', OneHotEncoder(), cat_attribs)
])

In [19]:
housing_prepared = full_pipeline.fit_transform(housing)

In [20]:
housing_prepared[0:5, :]

array([[-1.25390838e+00,  1.10757958e+00, -1.79230424e+00,
         3.43864740e+00,  3.19311274e+00,  2.57514612e+00,
         3.36084902e+00,  6.98297880e-01,  1.00672992e-01,
        -5.83111742e-02, -4.42046121e-01,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  1.00000000e+00,
         0.00000000e+00],
       [ 1.31649014e+00, -7.98632580e-01, -1.23670690e+00,
         2.35239297e+00,  2.50973879e+00,  1.63187609e+00,
         2.52702704e+00, -9.52232199e-01, -4.64530862e-02,
        -7.68056733e-02, -1.46426927e-01,  0.00000000e+00,
         1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 6.58946332e-01, -7.79898308e-01,  6.68198294e-01,
        -8.37654600e-01, -7.74252672e-01, -5.36771569e-01,
        -7.45724219e-01, -8.28020185e-01, -6.53605305e-01,
         6.74317450e-02,  7.62848976e-01,  1.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 1.14712279e+00, -1.31382505e

In [21]:
housing

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
1380,-122.09,38.00,6.0,10191.0,1882.0,4377.0,1789.0,5.2015,NEAR BAY
12294,-116.93,33.93,13.0,7804.0,1594.0,3297.0,1469.0,2.0549,INLAND
7387,-118.25,33.97,37.0,794.0,210.0,814.0,213.0,2.2917,<1H OCEAN
14454,-117.27,32.83,39.0,1877.0,426.0,805.0,409.0,3.8750,NEAR OCEAN
2927,-119.01,35.36,24.0,1941.0,484.0,1277.0,435.0,1.0560,INLAND
...,...,...,...,...,...,...,...,...,...
19391,-120.85,37.78,25.0,421.0,NaN,303.0,106.0,2.2679,INLAND
15393,-116.90,33.22,11.0,4132.0,773.0,2012.0,703.0,3.1906,<1H OCEAN
9143,-117.96,34.48,32.0,1896.0,342.0,806.0,299.0,4.5769,INLAND
17679,-121.84,37.32,22.0,3015.0,581.0,2491.0,530.0,4.3419,<1H OCEAN
